# 10.10 · 多模态学习 / Multimodal Learning (CLIP)

> **课程定位 / Where this fits**
> 第 10 课，**Part 10 · 计算机视觉**。
> Lesson 10, **Part 10 · Computer Vision**.
>
> 前面模型只"看"图。但人理解世界是**多模态**的——图像、文字、声音一起。2021 年的 **CLIP** 是里程碑：用**对比学习**把**图像和它的文字描述**对齐到**同一个向量空间**，从此"一张猫图"和"a photo of a cat"在空间里彼此靠近。这带来惊人的**零样本(zero-shot)分类**：不需为新任务训练，只要把类别名写成文字就能分类。CLIP 也是 DALL·E、Stable Diffusion 等的基石。本课用**合成图文数据从零实现对比学习**，并**亲眼见证零样本识别没见过的组合**。
> Earlier models only "see" images. But humans understand the world **multimodally** — image, text, sound together. **CLIP** (2021) is a landmark: it uses **contrastive learning** to align **images with their text descriptions** in **one shared space**, so "a cat photo" and "a photo of a cat" sit close. This enables striking **zero-shot classification**: no training for new tasks — just write the class names as text. CLIP also underpins DALL·E, Stable Diffusion, etc. We implement contrastive learning from scratch on synthetic image-text data and **witness zero-shot recognition of unseen combinations**.
>
> 💼 **实战/面试视角**："CLIP 怎么训练 / 对比学习 / 零样本为什么可行 / 图文检索" 是多模态/大模型岗热点。
> 💼 **Practical/interview angle:** "how CLIP trains / contrastive learning / why zero-shot works / image-text retrieval" — multimodal/LLM hot topics.

> 📐 **符号约定 / Notation**
> - 图像编码器 / 文本编码器 —— 各自把图/文映射成向量 / image & text encoders → vectors
> - 共享空间 —— 图与文嵌入在同一向量空间可比 / shared embedding space
> - 温度 $\tau$ —— 对比损失的缩放系数 / temperature in contrastive loss

> 💡 **面试相关 / Interview-relevant**
> - "CLIP 的对比学习目标(对角线对齐)"（出镜率 ★★★★★）
> - "零样本分类怎么做(类别名当文字)"（★★★★★）
> - "为什么能泛化到没见过的概念"（★★★★，共享语义空间/组合性）
> - "图文检索 / 多模态应用"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解多模态与 CLIP 的核心思想（共享空间对齐）。
   Understand multimodal learning and CLIP's core (shared-space alignment).
2. **从零实现对比学习**，看相似度矩阵对角线"点亮"。
   Implement contrastive learning from scratch; watch the similarity diagonal "light up."
3. 亲手做**零样本分类**，识别**训练时没见过的组合**。
   Do zero-shot classification, recognizing **unseen combinations**.
4. 了解真实 CLIP 与多模态应用。
   Know real CLIP and multimodal applications.

## 目录 / TOC
1. [多模态与 CLIP 的思想 ⭐](#1)
2. [合成图文数据 + 对比学习（从零）⭐](#2)
3. [零样本分类：识别没见过的组合 ⭐](#3)
4. [真实 CLIP 与应用 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 多模态与 CLIP 的思想 ⭐ / Multimodal Learning & CLIP

**多模态**指模型同时处理多种类型的数据（图像+文本+音频…）。CLIP 处理**图像+文本**两种，核心思路：
**Multimodal** means handling multiple data types (image+text+audio…). CLIP handles **image+text**, with this idea:
- 用一个**图像编码器**把图片变成向量，用一个**文本编码器**把文字变成向量。
  An **image encoder** turns pictures into vectors; a **text encoder** turns text into vectors.
- 训练目标：让**配对的(图, 文)** 的两个向量在空间里**靠近**，**不配对的远离**。这就是**对比学习(contrastive learning)**。
  Training goal: make vectors of **matched (image, text)** pairs **close**, and **mismatched** ones far. This is **contrastive learning**.
- 训练用海量"图片+它的网络配文"——这种数据**免费且无需人工标注**，能轻松搜集上亿对。
  Trained on massive "image + its web caption" pairs — **free, no manual labels**, scalable to hundreds of millions.

**为什么这带来零样本能力**（面试核心）：图和文在**同一空间**对齐后，要分类一张图，只需把候选类别名写成文字（"a photo of a dog"），编码成向量，**看图向量离哪个类别文字向量最近**即可——**完全不用在该任务上训练**。换新类别？换个文字就行。
**Why this gives zero-shot** (interview core): once image and text live in **one aligned space**, to classify an image you just write candidate class names as text ("a photo of a dog"), encode them, and **see which text vector the image is closest to** — **no task-specific training**. New classes? Just new text.


<a id="2"></a>
## 2. 合成图文数据 + 对比学习（从零）⭐ / Synthetic Image-Text + Contrastive Learning

真实 CLIP 用上亿网络图文对（需海量算力）。为了**离线、自包含**地讲透机制，我们造一个可控的合成多模态数据集：
Real CLIP uses hundreds of millions of web pairs (huge compute). To teach the mechanism **offline and self-contained**, we build a controllable synthetic multimodal dataset:
- **"图像"**：32×32 彩色图，画一个**形状**(圆/方/三角)，用某种**颜色**(红/绿/蓝)，随机位置大小+噪声。
  **"Images":** 32×32 color images showing a **shape** (circle/square/triangle) in a **color** (red/green/blue), random position/size + noise.
- **"文本"**：用一个**属性向量**表示描述——`[圆,方,三角, 红,绿,蓝]`（哪个形状+哪个颜色置 1）。这是"文字描述"的极简代理（真实 CLIP 用 Transformer 编码真实句子）。
  **"Text":** an **attribute vector** as the description — `[circle,square,triangle, red,green,blue]` (1s for the shape & color). A minimal proxy for a caption (real CLIP encodes real sentences with a Transformer).

共 3×3=9 种"形状-颜色"组合。下面生成数据并可视化几个(图, 文)对。
There are 3×3=9 shape-color combinations. Let's generate data and visualize a few (image, text) pairs.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
sns.set_theme(style="white")

shapes = ["circle", "square", "triangle"]; colors = ["red", "green", "blue"]
combos = [(s, c) for s in shapes for c in colors]                # 9 种组合 / 9 combinations

def draw(shape, color, H=32, W=32, rng=None):
    rng = rng or np.random
    img = np.abs(rng.normal(0, 0.05, (3, H, W))).astype(np.float32)   # 噪声背景 / noisy background
    cy, cx = rng.randint(11, 21), rng.randint(11, 21); r = rng.randint(6, 9)   # 随机位置/大小 / random pos/size
    yy, xx = np.mgrid[0:H, 0:W]
    if shape == "circle":   m = (xx-cx)**2 + (yy-cy)**2 <= r**2
    elif shape == "square": m = (np.abs(xx-cx) <= r) & (np.abs(yy-cy) <= r)
    else:                   m = (yy-(cy-r) >= 0) & (yy-(cy+r) <= 0) & (np.abs(xx-cx) <= (cy+r-yy)*0.9)  # 三角形 / triangle
    ch = {"red":0, "green":1, "blue":2}[color]
    for c in range(3): img[c][m] = 0.1
    img[ch][m] = 1.0                                             # 对应颜色通道点亮 / set the color channel
    return np.clip(img, 0, 1)

def attr(shape, color):                                          # "文本" = 属性向量 / "text" = attribute vector
    a = np.zeros(6, np.float32); a[shapes.index(shape)] = 1; a[3+colors.index(color)] = 1; return a

rng = np.random.RandomState(0)
fig, axes = plt.subplots(1, 5, figsize=(13, 2.9))
for ax, (s, c) in zip(axes, combos[:5]):
    ax.imshow(draw(s, c, rng=rng).transpose(1,2,0)); ax.axis("off")
    ax.set_title(f"{c} {s}\n文本属性:{attr(s,c).astype(int)}", fontsize=8)
fig.suptitle("合成图文对: 图像(形状+颜色) ↔ 文本属性向量[圆,方,三角,红,绿,蓝]"); plt.tight_layout(); plt.show()
print("9 种'形状-颜色'组合; 每张图配一个属性向量当'文字描述'(真实CLIP用Transformer编码真句子)")


**对比学习怎么训练**：一个 batch 里有 N 个(图, 文)对。算出 N 个图向量和 N 个文向量，两两点积得到 **N×N 相似度矩阵**。目标：**对角线(正确配对)相似度最高**，其它(错配)低。对每一行做 softmax + 交叉熵（让正确列概率最大），就是 CLIP 的对比损失(InfoNCE)。
**How contrastive training works:** a batch has N (image, text) pairs. Compute N image vectors and N text vectors; pairwise dot products give an **N×N similarity matrix**. Goal: **the diagonal (correct pairs) has the highest similarity**, off-diagonal low. Softmax + cross-entropy per row (correct column should win) is CLIP's contrastive loss (InfoNCE).

我们故意**只用 6 种组合训练**，留 3 种组合(`红三角/绿圆/蓝方`)**完全不参与训练**，留给 §3 做零样本。
We deliberately **train on only 6 combinations**, holding out 3 (`red triangle / green circle / blue square`) **entirely** for zero-shot in §3.


In [ ]:
heldout = [("triangle","red"), ("circle","green"), ("square","blue")]   # 留作零样本的3种组合 / held-out combos
train_combos = [c for c in combos if c not in heldout]                   # 训练用6种 / 6 training combos

def gen(combo_list, per, rng):
    X, lab = [], []
    for combo in combo_list:
        for _ in range(per): X.append(draw(*combo, rng=rng)); lab.append(combos.index(combo))
    return torch.tensor(np.array(X)), torch.tensor(lab)
Xtr, ytr = gen(train_combos, 80, rng)                                    # 训练图(只含6种组合) / training images
all_attr = torch.tensor(np.array([attr(*c) for c in combos]))           # 9 种组合的属性向量 / all 9 attr vectors

class ImgEnc(nn.Module):                                                 # 图像编码器 / image encoder
    def __init__(s, d=32):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
                              nn.Flatten(), nn.Linear(32, d))
    def forward(s, x): e = s.net(x); return e / e.norm(dim=-1, keepdim=True)   # L2归一化 / L2-normalize
class TxtEnc(nn.Module):                                                 # 文本(属性)编码器 / text encoder
    def __init__(s, d=32):
        super().__init__(); s.net = nn.Sequential(nn.Linear(6,32), nn.ReLU(), nn.Linear(32,d))
    def forward(s, a): e = s.net(a); return e / e.norm(dim=-1, keepdim=True)

# 一张代表图/组合 用于画相似度矩阵 / one representative image per training combo for the matrix
rep_imgs = torch.tensor(np.array([draw(*c, rng=np.random.RandomState(99)) for c in train_combos]))
train_attr = torch.tensor(np.array([attr(*c) for c in train_combos]))
def sim_matrix(ie, te):
    with torch.no_grad(): return (ie(rep_imgs) @ te(train_attr).T).numpy()  # 6×6 图-文相似度 / image-text similarity

torch.manual_seed(0); ie, te = ImgEnc(), TxtEnc()
before = sim_matrix(ie, te)                                              # 训练前 / before training
# 训练对比学习 / train contrastive learning
train_idx = {combos.index(c): i for i, c in enumerate(train_combos)}
ytr_local = torch.tensor([train_idx[int(y)] for y in ytr])
dl = DataLoader(TensorDataset(Xtr, ytr_local), batch_size=64, shuffle=True)
opt = torch.optim.Adam(list(ie.parameters())+list(te.parameters()), 1e-3); ce = nn.CrossEntropyLoss()
for ep in range(20):
    ie.train(); te.train()
    for xb, yb in dl:
        opt.zero_grad()
        logits = ie(xb) @ te(train_attr).T / 0.07        # 图向量·文向量/温度 = 相似度logits / similarity logits
        ce(logits, yb).backward(); opt.step()            # 让每张图匹配到正确的文本 / match image→correct text
after = sim_matrix(ie, te)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, M, t in [(axes[0], before, "训练前: 杂乱无规律"), (axes[1], after, "训练后: 对角线点亮(配对对齐)")]:
    im = ax.imshow(M, cmap="viridis"); ax.set_xlabel("文本(属性)"); ax.set_ylabel("图像")
    ax.set_xticks(range(6)); ax.set_yticks(range(6))
    ax.set_xticklabels([f"{c[1][:1]}{c[0][:2]}" for c in train_combos], fontsize=7)
    ax.set_yticklabels([f"{c[1][:1]}{c[0][:2]}" for c in train_combos], fontsize=7)
    ax.set_title(t); plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()
print("对比学习: 让(图,文)配对的相似度最高 → 相似度矩阵对角线'点亮'")
print("这就是 CLIP 的训练目标: 配对拉近, 错配推远(InfoNCE)")


<a id="3"></a>
## 3. 零样本分类：识别没见过的组合 ⭐ / Zero-Shot: Recognizing Unseen Combinations

现在见证 CLIP 的魔法。训练时模型**从未见过** `红三角/绿圆/蓝方` 这三种组合。但它见过红色(在红圆/红方里)、见过三角(在绿三角/蓝三角里)……即它学到了**形状**和**颜色**这两种**可组合的属性**在共享空间里的表示。
Now the magic. The model **never saw** the combos `red triangle / green circle / blue square` in training. But it saw red (in red circles/squares), saw triangles (in green/blue triangles)… i.e. it learned **composable attributes** (shape and color) in the shared space.

**零样本做法**：对一张没见过组合的图，把它编码成向量，再把**所有 9 种组合的文字属性**编码成向量，**看图离哪个文字最近**就预测哪个——哪怕那个组合从没在训练中出现过。
**Zero-shot recipe:** encode an unseen-combo image, encode the **text attributes of all 9 combos**, and predict **the nearest text** — even combos never seen in training.


In [ ]:
Xte, yte = gen(heldout, 40, np.random.RandomState(7))                   # 未见组合的测试图 / unseen-combo test images
ie.eval(); te.eval()
with torch.no_grad():
    concept_all = te(all_attr)                                           # 全部9种组合的文本向量 / all 9 text vectors
    sim_all = ie(Xte) @ concept_all.T                                    # 图 vs 9个文本 / image vs 9 texts
    pred_all = sim_all.argmax(1)
acc_all = (pred_all == yte).float().mean().item()
# 限定在 3 个未见组合之间分类 / classify only among the 3 held-out combos
ho_idx = [combos.index(c) for c in heldout]
with torch.no_grad():
    sim3 = ie(Xte) @ te(torch.tensor(np.array([attr(*c) for c in heldout]))).T
    pred3 = sim3.argmax(1); yte3 = torch.tensor([ho_idx.index(int(y)) for y in yte])
acc3 = (pred3 == yte3).float().mean().item()

# 可视化几张未见组合图的零样本预测 / visualize zero-shot predictions on unseen combos
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
for ax, j in zip(axes, np.linspace(0, len(Xte)-1, 6).astype(int)):
    ax.imshow(Xte[j].permute(1,2,0)); ax.axis("off")
    p = combos[pred_all[j]]; t = combos[int(yte[j])]
    ok = (pred_all[j]==yte[j])
    ax.set_title(f"真:{t[0][:3]}/{t[1][:1]}\n猜:{p[0][:3]}/{p[1][:1]}", fontsize=8, color="green" if ok else "red")
fig.suptitle("零样本: 这些'形状-颜色'组合训练时从未出现, 仍能靠属性组合识别"); plt.tight_layout(); plt.show()
print(f"零样本(在3个未见组合之间分类): 准确率 = {acc3:.3f}  (随机=1/3=0.33) ← 模型从没见过这些组合!")
print(f"零样本(在全部9个组合中分类):   准确率 = {acc_all:.3f}  (随机=1/9=0.11)")
print("含义: 模型把'形状'和'颜色'学成可组合的属性 → 能识别没见过的新组合(组合泛化)")
print("注: 全9类更难(未见组合会和相似的已见组合竞争); 真实CLIP靠海量数据+语言泛化缓解")


<a id="4"></a>
## 4. 真实 CLIP 与应用 + 小结 ⭐ / Real CLIP & Applications

我们的合成版抓住了核心机制。真实 **CLIP** 的关键差异：
Our synthetic version captures the core. Real **CLIP** differs in:
- **数据规模**：4 亿张"网络图片+配文"对(无需人工标注)。规模是零样本能力的关键。
  **Data scale:** 400M "web image + caption" pairs (no manual labels). Scale is key to zero-shot power.
- **文本编码器是 Transformer**：编码**任意自然语言**句子（不是我们的固定属性向量），所以能对**任意类别名**零样本（开放词表）。
  **Text encoder is a Transformer:** encodes **arbitrary natural language** (not fixed attribute vectors), enabling zero-shot for **any class name** (open vocabulary).
- **对称对比损失**：图→文 和 文→图 两个方向都算（我们只做了图→文一个方向）。
  **Symmetric loss:** both image→text and text→image (we did only image→text).
- **prompt 工程**：用"a photo of a {类别}"这样的模板，零样本效果更好。
  **Prompt engineering:** templates like "a photo of a {class}" improve zero-shot.

**多模态应用**（面试可举）：零样本图像分类、图文检索(以文搜图/以图搜文)、为 **DALL·E/Stable Diffusion** 等文生图模型提供文本-图像对齐、给大模型(LLaVA 等)装上"眼睛"。
**Applications:** zero-shot classification, image-text retrieval, providing text-image alignment for **DALL·E/Stable Diffusion**, giving LLMs "eyes" (e.g. LLaVA).

```
多模态: 同时处理图/文/音等; CLIP 处理图+文
CLIP: 图像编码器+文本编码器 → 对比学习把(图,文)配对对齐到同一空间(对角线点亮)
对比损失 InfoNCE: 配对相似度最高, 错配低; 用海量网络图文对(免费无标注)
零样本: 类别名写成文字→编码→图离哪个文字近就预测哪个; 无需为新任务训练
泛化: 学到可组合的属性/共享语义空间 → 识别没见过的组合/类别
真实CLIP: 4亿图文对 + Transformer文本编码器(开放词表) + 对称损失 + prompt
应用: 零样本分类/图文检索/文生图(DALL·E,SD)/多模态大模型(LLaVA)
```

### 💡 面试速查 / Interview cheat-sheet
1. **CLIP 训练**: 对比学习, (图,文)配对相似度最高(相似度矩阵对角线)。
   CLIP training: contrastive, matched (image,text) pairs score highest (diagonal).
2. **零样本**: 把类别名当文字编码, 图离哪个近选哪个, 无需训练。
   Zero-shot: encode class names as text, pick nearest to the image, no training.
3. **为什么泛化**: 图文对齐到共享语义空间 + 语言的组合性。
   Why it generalizes: shared semantic space + compositionality of language.
4. **数据**: 海量网络图文对, 免费无需人工标注。
   Data: massive web image-text pairs, free, no manual labels.
5. **应用**: 检索 / 文生图引导(SD) / 多模态大模型。
   Applications: retrieval / text-to-image guidance / multimodal LLMs.

### 下一节 / Next
**10.11 自监督学习**——CLIP 用图文对，那只有海量**无标注图片**呢？自监督学习从数据本身造"任务"(如对比同一图的两个增强视角)，**不用任何人工标签**就学到强大特征。这是现代表示学习的前沿，也是 Part 10 的收尾。
**10.11 Self-Supervised Learning** — CLIP used image-text pairs; what about just massive **unlabeled images**? Self-supervised learning invents tasks from the data itself (e.g. contrasting two augmented views of one image), learning strong features **with no human labels**. The frontier of representation learning, and the finale of Part 10.
